# 01 - Agent 基础教程 (Agent Fundamentals)

## 学习目标

完成本教程后，您将能够：

1. **理解 Agent 核心概念** - 掌握智能体的定义、特性和理论基础
2. **掌握 Agent 角色系统** - 了解不同角色的职责和应用场景
3. **理解状态机模型** - 掌握 Agent 生命周期和状态转换
4. **配置和创建 Agent** - 学会使用配置类和工厂函数
5. **实现对话交互** - 掌握单轮和多轮对话的实现方法
6. **自定义 Agent** - 学会继承 BaseAgent 创建专用智能体
7. **理解 ReAct 模式** - 掌握推理-行动循环的实现

---

## 理论基础

### 什么是智能体 (Agent)？

在人工智能领域，**智能体 (Agent)** 是一个能够感知环境、做出决策并采取行动的自主计算实体。

#### 智能体的四大特性

| 特性 | 英文 | 描述 |
|------|------|------|
| **自主性** | Autonomy | 无需外部干预即可独立运行 |
| **反应性** | Reactivity | 能够感知环境变化并做出响应 |
| **主动性** | Pro-activeness | 能够主动采取行动实现目标 |
| **社会性** | Social ability | 能够与其他智能体或人类交互 |

#### 数学建模

智能体可以形式化为一个四元组：$A = \langle S, P, M, \pi \rangle$

- $S$: 内部状态空间
- $P$: 感知函数 $P: Environment \to Observation$
- $M$: 记忆模块 $M: History \to Context$
- $\pi$: 策略函数 $\pi: S \times O \to Action$

In [ ]:
# 环境设置
import sys
import asyncio
from pathlib import Path
from datetime import datetime

src_path = Path.cwd().parent / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from agent_base import (
    AgentRole, AgentState, AgentConfig, AgentResponse,
    BaseAgent, SimpleAgent, ReActAgent, MockLLM, create_agent
)

def run_async(coro):
    """运行异步函数"""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    return loop.run_until_complete(coro)

print('=' * 60)
print('多智能体系统 - Agent 基础教程')
print('=' * 60)
print(f'时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('环境设置完成！')

---

## 第一部分：Agent 角色系统

### 1.1 角色类型

| 角色 | 描述 | 典型用途 |
|------|------|----------|
| ASSISTANT | 通用助手 | 问答、任务执行 |
| CRITIC | 评论者 | 代码审查、质量检查 |
| MANAGER | 管理者 | 任务分配、流程控制 |
| RESEARCHER | 研究者 | 信息收集、数据分析 |
| CODER | 编码者 | 代码生成、调试 |
| DEBATER | 辩论者 | 观点论证、决策支持 |

In [ ]:
# 1.1 查看所有角色
print('=' * 60)
print('Agent 角色类型 (AgentRole)')
print('=' * 60)

role_desc = {
    'assistant': '通用助手 - 处理日常任务',
    'critic': '评论者 - 评估和审查',
    'manager': '管理者 - 协调资源',
    'researcher': '研究者 - 信息分析',
    'coder': '编码者 - 代码生成',
    'debater': '辩论者 - 论证观点',
}

for i, role in enumerate(AgentRole, 1):
    desc = role_desc.get(role.value, '')
    print(f'{i}. {role.name:12} | "{role.value:10}" | {desc}')

print(f'\n共 {len(list(AgentRole))} 种角色')

In [ ]:
# 1.2 角色使用示例
print('\n角色使用示例:')
print('-' * 40)

examples = [
    (AgentRole.ASSISTANT, '小助手', '日常问答'),
    (AgentRole.CODER, '程序员', '代码生成'),
    (AgentRole.CRITIC, '审查员', '代码审查'),
]

for role, name, use in examples:
    print(f'角色: {role.value}, 名称: {name}, 用途: {use}')

---

## 第二部分：Agent 状态机

### 2.1 状态说明

| 状态 | 描述 | 可转换到 |
|------|------|----------|
| IDLE | 空闲，等待输入 | THINKING, TERMINATED |
| THINKING | 处理中 | SPEAKING, ERROR |
| SPEAKING | 输出中 | IDLE, ERROR |
| ERROR | 错误状态 | IDLE |
| TERMINATED | 已终止 | (无) |

In [ ]:
# 2.1 查看所有状态
print('=' * 60)
print('Agent 状态 (AgentState)')
print('=' * 60)

for state in AgentState:
    print(f'  {state.name:12} -> "{state.value}"')

In [ ]:
# 2.2 状态转换验证
print('\n状态转换规则验证:')
print('-' * 60)

transitions = [
    (AgentState.IDLE, AgentState.THINKING, '开始处理', True),
    (AgentState.THINKING, AgentState.SPEAKING, '生成响应', True),
    (AgentState.SPEAKING, AgentState.IDLE, '完成输出', True),
    (AgentState.IDLE, AgentState.TERMINATED, '终止', True),
    (AgentState.TERMINATED, AgentState.IDLE, '恢复(应失败)', False),
    (AgentState.ERROR, AgentState.IDLE, '错误恢复', True),
]

for from_s, to_s, desc, expected in transitions:
    actual = from_s.can_transition_to(to_s)
    status = '✓' if actual == expected else '✗'
    print(f'{status} {from_s.value:10} -> {to_s.value:10} ({desc})')

---

## 第三部分：Agent 配置

### 3.1 配置参数

| 参数 | 类型 | 默认值 | 描述 |
|------|------|--------|------|
| name | str | (必填) | Agent 名称 |
| role | AgentRole | ASSISTANT | 功能角色 |
| system_prompt | str | "" | 系统提示词 |
| temperature | float | 0.7 | 采样温度 (0-2) |
| max_tokens | int | 1000 | 最大输出 token |
| capabilities | Set[str] | {} | 能力标签 |

In [ ]:
# 3.1 创建配置
print('=' * 60)
print('Agent 配置示例')
print('=' * 60)

# 最简配置
simple_config = AgentConfig(name='SimpleBot')
print('\n[最简配置]')
print(f'  名称: {simple_config.name}')
print(f'  角色: {simple_config.role}')
print(f'  温度: {simple_config.temperature}')

# 完整配置
full_config = AgentConfig(
    name='高级助手',
    role=AgentRole.ASSISTANT,
    system_prompt='你是一个专业的AI助手。',
    temperature=0.7,
    max_tokens=2000,
    capabilities={'chat', 'code_review'}
)
print('\n[完整配置]')
print(f'  名称: {full_config.name}')
print(f'  角色: {full_config.role.value}')
print(f'  能力: {full_config.capabilities}')

In [ ]:
# 3.2 配置验证
print('\n[配置验证]')
print('-' * 40)

# 测试无效配置
try:
    AgentConfig(name='')  # 空名称
except ValueError as e:
    print(f'✓ 空名称验证: 捕获异常')

try:
    AgentConfig(name='Test', temperature=3.0)  # 温度超范围
except ValueError as e:
    print(f'✓ 温度验证: 捕获异常')

print('\n配置验证正常！')

---

## 第四部分：创建和使用 Agent

In [ ]:
# 4.1 使用工厂函数创建 Agent
print('=' * 60)
print('创建 Agent')
print('=' * 60)

mock_llm = MockLLM(responses=[
    '你好！我是AI助手。',
    '我可以帮助你完成各种任务。',
    '这是一个很好的问题！',
])

agent = create_agent(
    name='小助手',
    role='assistant',
    system_prompt='你是一个友好的AI助手。',
    llm=mock_llm
)

print(f'Agent 创建成功！')
print(f'  名称: {agent.name}')
print(f'  ID: {agent.id}')
print(f'  状态: {agent.state}')
print(f'  活跃: {agent.is_active}')

In [ ]:
# 4.2 单轮对话
print('\n' + '=' * 60)
print('单轮对话演示')
print('=' * 60)

async def single_turn():
    user_input = '你好，请介绍一下你自己'
    print(f'\n用户: {user_input}')
    response = await agent.step(user_input)
    print(f'Agent: {response.content}')
    return response

run_async(single_turn())

In [ ]:
# 4.3 多轮对话
print('\n' + '=' * 60)
print('多轮对话演示')
print('=' * 60)

async def multi_turn():
    agent.reset()
    conversations = ['你好！', '你能做什么？']
    
    for i, msg in enumerate(conversations, 1):
        print(f'\n--- 第 {i} 轮 ---')
        print(f'用户: {msg}')
        response = await agent.step(msg)
        print(f'Agent: {response.content}')

run_async(multi_turn())

In [ ]:
# 4.4 查看对话历史
print('\n' + '=' * 60)
print('对话历史')
print('=' * 60)

history = agent.get_history()
print(f'\n共 {len(history)} 条消息:')
for i, msg in enumerate(history):
    role = msg.get('role', 'unknown')
    content = msg.get('content', '')[:50]
    print(f'{i+1}. [{role}] {content}...')

---

## 第五部分：Agent 生命周期管理

In [ ]:
# 5.1 重置 Agent
print('=' * 60)
print('Agent 重置')
print('=' * 60)

print(f'重置前历史长度: {len(agent.get_history())}')
agent.reset()
print(f'重置后历史长度: {len(agent.get_history())}')
print('✓ Agent 已重置')

In [ ]:
# 5.2 终止 Agent
print('\n' + '=' * 60)
print('Agent 终止')
print('=' * 60)

temp_agent = create_agent('临时Agent', llm=MockLLM(responses=['test']))
print(f'终止前状态: {temp_agent.state}, 活跃: {temp_agent.is_active}')
temp_agent.terminate()
print(f'终止后状态: {temp_agent.state}, 活跃: {temp_agent.is_active}')
print('⚠ 终止后的 Agent 无法恢复')

---

## 第六部分：自定义 Agent

In [ ]:
# 6.1 创建 EchoAgent
print('=' * 60)
print('自定义 Agent: EchoAgent')
print('=' * 60)

class EchoAgent(BaseAgent):
    """回声 Agent - 将输入转为大写返回"""
    
    async def think(self, input_text: str) -> str:
        return f'收到: "{input_text}"'
    
    async def act(self, thought: str) -> str:
        import re
        match = re.search(r'"(.+?)"', thought)
        if match:
            return f'ECHO: {match.group(1).upper()}'
        return 'ECHO: [无法解析]'

echo_config = AgentConfig(name='EchoBot')
echo_agent = EchoAgent(echo_config)
print(f'创建 EchoAgent: {echo_agent.name}')

In [ ]:
# 6.2 测试 EchoAgent
async def test_echo():
    messages = ['hello world', 'Python is awesome', '多智能体系统']
    print('\n测试 EchoAgent:')
    for msg in messages:
        response = await echo_agent.step(msg)
        print(f'  输入: {msg} -> 输出: {response.content}')

run_async(test_echo())

In [ ]:
# 6.3 创建 CalculatorAgent
print('\n' + '=' * 60)
print('自定义 Agent: CalculatorAgent')
print('=' * 60)

class CalculatorAgent(BaseAgent):
    """计算器 Agent - 执行数学运算"""
    
    async def think(self, input_text: str) -> str:
        import re
        pattern = r'(\d+(?:\.\d+)?)\s*([+\-*/])\s*(\d+(?:\.\d+)?)'
        match = re.search(pattern, input_text)
        if match:
            return f'CALC:{match.group(1)}:{match.group(2)}:{match.group(3)}'
        return 'ERROR:无法解析'
    
    async def act(self, thought: str) -> str:
        if thought.startswith('ERROR:'):
            return thought.replace('ERROR:', '错误: ')
        parts = thought.split(':')
        if len(parts) != 4:
            return '错误: 解析失败'
        _, a, op, b = parts
        a, b = float(a), float(b)
        ops = {'+': lambda x,y: x+y, '-': lambda x,y: x-y,
               '*': lambda x,y: x*y, '/': lambda x,y: x/y if y!=0 else float('inf')}
        return f'{a} {op} {b} = {ops[op](a, b)}'

calc_agent = CalculatorAgent(AgentConfig(name='Calculator'))

async def test_calc():
    exprs = ['10 + 5', '100 - 37', '6 * 7', '144 / 12']
    print('\n测试 CalculatorAgent:')
    for expr in exprs:
        response = await calc_agent.step(expr)
        print(f'  {expr} -> {response.content}')

run_async(test_calc())

---

## 第七部分：ReAct Agent

### ReAct 模式

**ReAct (Reasoning + Acting)** 将推理和行动交织：

```
Thought -> Action -> Observation -> Thought -> ... -> Final Answer
```

In [ ]:
# 7.1 创建 ReAct Agent
print('=' * 60)
print('ReAct Agent')
print('=' * 60)

react_llm = MockLLM(responses=[
    'Thought: 分析问题\nAction: search\nAction Input: Python',
    'Thought: 找到信息\nFinal Answer: Python 是高级编程语言。',
])

react_agent = create_agent(
    name='ReActBot',
    agent_type='react',
    llm=react_llm
)

print(f'创建 ReAct Agent: {react_agent.name}')
print(f'类型: {type(react_agent).__name__}')

In [ ]:
# 7.2 测试 ReAct Agent
async def test_react():
    print('\n测试 ReAct Agent:')
    response = await react_agent.step('什么是 Python？')
    print(f'问题: 什么是 Python？')
    print(f'响应: {response.content}')

run_async(test_react())

---

## 总结

本教程涵盖了 Agent 基础的核心内容：

1. **AgentRole**: 定义 Agent 的功能角色
2. **AgentState**: 管理 Agent 的生命周期状态
3. **AgentConfig**: 配置 Agent 的各项参数
4. **create_agent()**: 使用工厂函数快速创建 Agent
5. **对话交互**: 单轮和多轮对话的实现
6. **生命周期管理**: reset() 和 terminate() 操作
7. **自定义 Agent**: 继承 BaseAgent 创建专用 Agent
8. **ReAct Agent**: 推理-行动循环模式

### 下一步

- 学习 **02_MultiAgentDebate_tutorial.ipynb** 了解辩论式多智能体
- 学习 **03_CollaborativeAgents_tutorial.ipynb** 了解协作式多智能体